# CSE 144 Final Project — Google Colab runner

**Runtime → Change runtime type → GPU** (T4 or better). All logic lives in `src/*.py`.
This notebook installs deps, wires `Data/`, trains, and writes **`submission.csv` (1036 rows)**.

**Checkpoints are written directly to Google Drive** so an ended/crashed Colab session can't wipe
them — you can resume inference in a fresh session.

**Build order (gate each layer on OOF improvement):**
1. **ConvNeXt-S 5-fold → baseline OOF + first accepted submission**  ← *this run*
2. + resolution bump → 3. + TTA → 4. + MixUp/EMA → 5. + EffV2-S & ViT ensemble → 6. + seed averaging

### 1. GPU check + install deps

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — Runtime>Change runtime type>GPU')
!pip install -q timm==1.0.27 scikit-learn pyyaml

### 2. Get the code + Drive paths
Clones the **updated** repo each fresh session (picks up the latest pushed config). Edit the two
Drive paths. `CKPT_DRIVE` is where checkpoints persist between sessions.

In [ ]:
import os, glob, subprocess

REPO_URL = 'https://github.com/Sequoiabm/cse144-final.git'
PROJ = '/content/cse144-final'
if not os.path.isdir(PROJ):
    subprocess.run(['git', 'clone', REPO_URL, PROJ], check=True)
%cd {PROJ}
!git pull -q          # ensure latest config.yaml even if dir already existed

from google.colab import drive
drive.mount('/content/drive')

DATA_ON_DRIVE = '/content/drive/MyDrive/CSE144/Data'          # <-- EDIT: must contain train/, test/
CKPT_DRIVE    = '/content/drive/MyDrive/CSE144/checkpoints'   # <-- EDIT: where weights persist
os.makedirs(CKPT_DRIVE, exist_ok=True)

if not os.path.isdir('Data'):
    assert os.path.isdir(DATA_ON_DRIVE), f'Upload Data to {DATA_ON_DRIVE} (train/, test/, sample_submission.csv)'
    os.symlink(DATA_ON_DRIVE, 'Data')

n_train, n_test = len(glob.glob('Data/train/*')), len(glob.glob('Data/test/*.jpg'))
print(f'cwd={os.getcwd()} | train classes={n_train} | test jpg={n_test} (expect 100, 1036)')
assert n_train == 100 and n_test == 1036, 'Fix Data/ paths before training'

### 3. Train ConvNeXt-S, 5 folds → checkpoints saved to Drive
`--out {CKPT_DRIVE}` writes each fold's raw+EMA ckpt + OOF logits straight to Drive as it finishes,
so even a mid-run disconnect keeps completed folds. Full 40 epochs/fold (early stop disabled),
`num_workers=2` — both come from the updated `config.yaml`. Expect ~1.5–2 h on a T4.

In [ ]:
!python src/train.py --backbone convnext_small --device cuda --out {CKPT_DRIVE}

### 3b. Baseline OOF accuracy (the number to log + beat)

In [ ]:
import numpy as np
z = np.load(f'{CKPT_DRIVE}/convnext_small/oof.npz'); cov = z['covered']
oof = float((z['logits'][cov].argmax(1) == z['labels'][cov]).mean())
print(f'ConvNeXt-S 5-fold BASELINE OOF accuracy: {oof:.4f} over {int(cov.sum())} samples')

### 4. Predict all 1036 test images → submission.csv
ConvNeXt-S only (the baseline), all 5 folds averaged + hflip TTA. Reads ckpts from Drive, so this
cell works in a fresh session too (run cells 1–2 first, skip training).

In [ ]:
!python src/predict.py --backbones convnext_small --ckpt-dir {CKPT_DRIVE} --out submission.csv
import pandas as pd
sub = pd.read_csv('submission.csv')
assert len(sub) == 1036 and list(sub.columns) == ['ID','Label'], (len(sub), list(sub.columns))
print('OK:', sub.shape); sub.head()

### 5. Download + submit
Download `submission.csv`, then upload at the
[competition Submit page](https://www.kaggle.com/competitions/ucsc-cse-144-spring-2026-final-project/submit).

In [ ]:
from google.colab import files
files.download('submission.csv')
# Optional Kaggle API (put kaggle.json on Drive first):
# !pip install -q kaggle && mkdir -p ~/.kaggle && cp /content/drive/MyDrive/.kaggle/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions submit -c ucsc-cse-144-spring-2026-final-project -f submission.csv -m "ConvNeXt-S 5-fold baseline"

---
### LATER — improvement layers (only after baseline is logged)
Resolution bump → TTA tuning → MixUp/EMA deltas → add EffV2-S & ViT (`config.ensemble`) → seed averaging.
Each `!python src/train.py --backbone <bk> --device cuda --out {CKPT_DRIVE}`, then re-run cell 4 with
`--backbones convnext_small,effv2s,vit_base`. Gate every addition on its OOF delta.